# 05 — GRACE Diagnostics

Reproduces paper **Fig. 10** (`GRACE_all_method_effects.png`),
**Fig. 11** (`cluster_hallucinating.png`), **Fig. 12**
(`gg_centric_rep_fragmentation.png`).


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from grace.analysis.load_results import load_summary_results
from grace.diagnostics.alignment import alignment_per_layer
from grace.diagnostics.fragmentation import pl_ra_correlation
from grace.diagnostics.pair_heatmap import per_pair_similarity

FIG_DIR = Path('Images'); FIG_DIR.mkdir(exist_ok=True)
MODELS = {'gemma2': 'google/gemma-2-2b-it', 'gemma3': 'google/gemma-3-27b-it',
          'llama3': 'meta-llama/Llama-3.3-70B-Instruct'}
CONCEPTS = sorted(p.stem for p in Path('concepts/gpt-5/extract').glob('*.json'))

## Fig. 10 — Per-(model, concept) Δsteerability vs. PV baseline for the 3 GRACE methods

For each (model, concept), look up the best-utility configuration for `pv`,
`unit_mean`, and `cluster`. Plot the per-method delta as a heatmap.


In [ ]:
def best_utility(model_name, concept, method):
    rows = [r for r in load_summary_results(model_name, concept, method=method) if r.get('mean_utility') is not None]
    return max((r['mean_utility'] for r in rows), default=None)

data = []
for tag, mname in MODELS.items():
    for c in CONCEPTS:
        pv = best_utility(mname, c, 'pv')
        if pv is None:
            continue
        for m in ['unit_mean', 'cluster']:
            v = best_utility(mname, c, m)
            if v is None:
                continue
            data.append({'model': tag, 'concept': c, 'method': m, 'delta': v - pv})
df = pd.DataFrame(data)
pivot = df.pivot_table(index=['model', 'concept'], columns='method', values='delta')
fig, ax = plt.subplots(figsize=(6, 12))
im = ax.imshow(pivot.values, cmap='RdBu_r', vmin=-5, vmax=5, aspect='auto')
ax.set_yticks(range(len(pivot)))
ax.set_yticklabels([f'{m}/{c}' for m, c in pivot.index], fontsize=6)
ax.set_xticks(range(pivot.shape[1])); ax.set_xticklabels(pivot.columns)
plt.colorbar(im, ax=ax, label='Δsteerability vs. pv')
plt.tight_layout(); plt.savefig(FIG_DIR / 'GRACE_all_method_effects.png', dpi=150); plt.show()

## Fig. 11 — Cluster diagnostic for `hallucinating`


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, model_name, title in [
    (axes[0], MODELS['gemma2'], 'gemma-2-2b: pair-similarity heatmap'),
    (axes[2], MODELS['gemma3'], 'gemma-3-27b: pair-similarity heatmap'),
]:
    sims = per_pair_similarity(model_name, 'hallucinating', 'response_avg')
    if sims:
        avg_sim = np.mean(np.stack([s.numpy() for s in sims.values()]), axis=0)
        im = ax.imshow(avg_sim, cmap='RdBu_r', vmin=-1, vmax=1)
        ax.set_title(title); plt.colorbar(im, ax=ax)

ax = axes[1]
for method, color in [('pv', 'C0'), ('cluster', 'C1')]:
    rows = [r for r in load_summary_results(MODELS['gemma2'], 'hallucinating', method=method) if r.get('mean_utility') is not None]
    if not rows:
        continue
    df = pd.DataFrame(rows).groupby('coef')['mean_utility'].max().reset_index()
    ax.plot(df.coef, df.mean_utility, '-o', label=method, color=color)
ax.set_title('hallucinating — gemma-2-2b: utility vs. coef'); ax.set_xlabel('coef'); ax.set_ylabel('best utility'); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / 'cluster_hallucinating.png', dpi=150); plt.show()

## Fig. 12 — Representational fragmentation diagnostic for `golden_gate_centric`

Left: PL/RA correlation across all (model, concept) pairs vs. constrained-search delta.
Right: layer profiles for `golden_gate_centric` on Gemma-2-2B (low correlation case).


In [ ]:
rows = []
for tag, mname in MODELS.items():
    for c in CONCEPTS:
        try:
            r, _, _ = pl_ra_correlation(mname, c)
        except FileNotFoundError:
            continue
        pv_unc = best_utility(mname, c, 'pv')
        if pv_unc is None: continue
        rows.append({'model': tag, 'concept': c, 'pl_ra_r': r, 'pv_unc': pv_unc})
frag = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(frag.pl_ra_r, frag.pv_unc, alpha=0.5)
axes[0].axvline(0.2, color='red', linestyle='--', label='fragmentation threshold')
axes[0].set_xlabel('Pearson correlation (𝒜_PL, 𝒜_RA)')
axes[0].set_ylabel('PV best utility')
axes[0].legend(); axes[0].set_title('PL/RA fragmentation diagnostic')

ax = axes[1]
_, A_pl, A_ra = pl_ra_correlation(MODELS['gemma2'], 'golden_gate_centric')
layers = sorted(set(A_pl) & set(A_ra))
ax.plot(layers, [A_pl[l] for l in layers], label='PL alignment')
ax.plot(layers, [A_ra[l] for l in layers], label='RA alignment')
ax.set_title('golden_gate_centric on gemma-2-2b: PL vs RA layer profiles')
ax.set_xlabel('layer'); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / 'gg_centric_rep_fragmentation.png', dpi=150); plt.show()